## TON IoT Data

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# import pandas as pd
# import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, classification_report
import matplotlib.pyplot as plt
# import torch.nn as nn
# import torch.nn.functional as F
# import torch
from torch.utils.data import DataLoader, TensorDataset
import math, time, os, datetime, psutil, gc
import random
import kagglehub
import warnings

%load_ext autoreload
%autoreload 2
from utility import *
from informer import *
from libs import *

/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Ingestion Step

In [2]:
os.listdir('../data/TON_IoT/')

['train_test_network.csv',
 'IoT_Garage_Door.csv',
 'IoT_Thermostat.csv',
 'Train_Test_IoT_Fridge.xlsx',
 'IoT_GPS_Tracker.csv',
 'IoT_Fridge.csv',
 'df_iot_os_linux.pkl',
 'IoT_Modbus.csv',
 'IoT_Weather.csv',
 'IoT_Motion_Light.csv']

In [3]:
file='../data/TON_IoT/Train_Test_IoT_Fridge.xlsx'

In [4]:
### IOT Fridge
# file = '../data/TON_IoT/IoT_Fridge.csv'
df= pd.read_excel(file)
df.head()
col_resp='label'
col = 'temp_condition'

,date,time,fridge_temperature,temp_condition,label,type
0,2019-04-25,19:19:40,9.00,high,1,ddos
1,2019-04-25,19:19:40,9.25,high,1,ddos
2,2019-04-25,19:19:45,12.65,high,1,ddos
3,2019-04-25,19:19:45,4.65,low,1,ddos
4,2019-04-25,19:19:55,12.65,high,1,ddos


In [5]:
df.shape
print(100*df['label'].sum()/df.shape[0])

(39944, 6)

62.44742639695574


## Data Preprocessing

In [6]:
from sklearn.preprocessing import LabelEncoder

print(df['temp_condition'].value_counts())
# --- Clean column ---
df[col] = df[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

# --- Check values ---
print(df[col].value_counts())

# --- Label Encoding ---
le_protocol = LabelEncoder()
df[col] = le_protocol.fit_transform(df[col])

print(df[col].value_counts())

temp_condition
high      13939
low       11005
high       6074
low        4662
high       2373
low        1891
Name: count, dtype: int64
temp_condition
high    22386
low     17558
Name: count, dtype: int64
temp_condition
0    22386
1    17558
Name: count, dtype: int64


In [7]:
# --- Clean date & time ---
df["date"] = df["date"].astype(str).str.strip()
df["time"] = df["time"].astype(str).str.strip()

# --- Create datetime ---
df["datetime"] = pd.to_datetime(
    df["date"] + " " + df["time"],
    # format="%d-%b-%y %H:%M:%S",
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# --- Clean response column ---
df[col_resp] = df[col_resp].astype(str).str.strip()

# --- Drop old columns ---
df.drop(columns=['date', 'time'], inplace=True)

# --- Extract features ---
dt = df["datetime"]

df["month"]   = dt.dt.month
df["day"]     = dt.dt.day
df["weekday"] = dt.dt.weekday
df["hour"]    = dt.dt.hour
df["minute"]  = dt.dt.minute
df["second"]  = dt.dt.second

In [8]:
df

,fridge_temperature,temp_condition,label,type,datetime,month,day,weekday,hour,minute,second
0,9.00,0,1,ddos,2019-04-25 19:19:40,4,25,3,19,19,40
1,9.25,0,1,ddos,2019-04-25 19:19:40,4,25,3,19,19,40
2,12.65,0,1,ddos,2019-04-25 19:19:45,4,25,3,19,19,45
3,4.65,1,1,ddos,2019-04-25 19:19:45,4,25,3,19,19,45
4,12.65,0,1,ddos,2019-04-25 19:19:55,4,25,3,19,19,55
...,...,...,...,...,...,...,...,...,...,...,...
39939,13.25,0,1,xss,2019-04-27 05:13:35,4,27,5,5,13,35
39940,4.35,1,1,xss,2019-04-27 05:13:36,4,27,5,5,13,36
39941,1.00,1,1,xss,2019-04-27 05:13:36,4,27,5,5,13,36
39942,10.25,0,1,xss,2019-04-27 05:13:36,4,27,5,5,13,36


In [9]:
df['fridge_temperature'] = df['fridge_temperature'].diff().fillna(0)

In [10]:
scaler = StandardScaler()
scaler.fit(df['fridge_temperature'].values.reshape(-1,1))
value = scaler.fit_transform(df['fridge_temperature'].values.reshape(-1,1))

StandardScaler()

In [11]:
# table = df.groupby(['month','day'])['type'].value_counts().unstack(fill_value=0)
df.pivot_table(index=['month', 'day'], columns='type', aggfunc='size', fill_value=0)

type       backdoor  ddos  injection  normal  password  ransomware   xss
month day                                                               
3     31          0     0          0    8942         0           0     0
4     1           0     0          0    1794         0           0     0
      2           0     0          0    4264         0           0     0
      25          0  5000       5000       0         0           0     0
      26          0     0          0       0      5000           0     0
      27          0     0          0       0         0           0  2042
      28       4086     0          0       0         0        2902     0
      29        914     0          0       0         0           0     0

In [12]:
# time_cols = ["month", "day", "weekday", "hour", "minute", "second"]
time_cols = ["month", "day", "weekday", "hour"]
X_mark = df[time_cols]
excluded = time_cols.copy() + ['label', 'type', 'datetime','date', 'time']

In [13]:
# Separate by data type (The "Clean" Way)
num_feature_cols = ['fridge_temperature']
cat_feature_cols = ['temp_condition']
# combined_df = df[[num_feature_cols, cat_feature_cols]]
X_num = df[num_feature_cols].values
X_cat = df[cat_feature_cols].values
X = pd.concat([df[num_feature_cols], df[cat_feature_cols]], axis=1)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df[col_resp] = le.fit_transform(df[col_resp])

mapping = dict(zip(le.transform(le.classes_), le.classes_))

y_label = df[col_resp].values

In [14]:
feature_metadata = {
    'num_cols': num_feature_cols,
    'cat_cols': cat_feature_cols,
    'cat_cardinalities': {col: df[col].nunique() for col in cat_feature_cols}
}

## Running and saving the models

In [15]:
## Setting Parameters
# data loading params
seq_len = 90
batch_size = 32

# Model training params
pred_len = 1
d_ff=512
d_model=512
num_epochs = 50

In [16]:
print(f"---Model Training, Validation and Testing Starts ---")

##################################
### 1. Split train test (Chronological)
print('Split Train, Val, Test')
N = len(X)
train_end = int(0.5 * N)
val_end   = int(0.6 * N)

X_train_full, X_val, X_test = X[:train_end], X[train_end:val_end], X[val_end:]
M_train_full, M_val, M_test = X_mark[:train_end], X_mark[train_end:val_end], X_mark[val_end:]
y_train_label, y_val_label, y_test_label = y_label[:train_end], y_label[train_end:val_end], y_label[val_end:]

## Masking to learn normality
train_mask = (y_label[:train_end] == 0)
X_train = X_train_full[train_mask]
M_train = M_train_full[train_mask]

# No masking. Learning labels
# X_train, M_train = X_train_full, M_train_full

##################################
### 2. Feature-Specific Processing
num_cols = feature_metadata['num_cols']
cat_cols = feature_metadata['cat_cols']

# --- Handle Numerical Features ---
if len(num_cols) > 0:
    print(f'Rescaling {len(num_cols)} numeric features')
    scaler = StandardScaler()
    # Use full training set to fit for better statistics
    X_train_num = scaler.fit_transform(X_train[num_cols]) 
    X_val_num = scaler.transform(X_val[num_cols])
    X_test_num = scaler.transform(X_test[num_cols])
else:
    X_train_num, X_val_num, X_test_num = None, None, None

# --- Handle Categorical Features ---
if len(cat_cols) > 0:
    print(f'Processing {len(cat_cols)} categorical features')
    X_train_cat = X_train[cat_cols].values.astype(int)
    X_val_cat = X_val[cat_cols].values.astype(int)
    X_test_cat = X_test[cat_cols].values.astype(int)
else:
    X_train_cat, X_val_cat, X_test_cat = None, None, None

##################################
### 3. Combine and Indexing
# Combine into a single array: [Numerical, Categorical]
processed_data = []
if X_train_num is not None: processed_data.append(X_train_num)
if X_train_cat is not None: processed_data.append(X_train_cat)
X_train_final = np.concatenate(processed_data, axis=1)

processed_data_val = []
if X_val_num is not None: processed_data_val.append(X_val_num)
if X_val_cat is not None: processed_data_val.append(X_val_cat)
X_val_final = np.concatenate(processed_data_val, axis=1)

processed_data_test = []
if X_test_num is not None: processed_data_test.append(X_test_num)
if X_test_cat is not None: processed_data_test.append(X_test_cat)
X_test_final = np.concatenate(processed_data_test, axis=1)

# Recalculate column indices for the Dataset class
# Numerical always comes first in our concatenation above
num_idxs = list(range(len(num_cols))) if len(num_cols) > 0 else []
cat_idxs = list(range(len(num_cols), len(num_cols) + len(cat_cols))) if len(cat_cols) > 0 else []

##################################
### 4. Data Loading
print('Data Loading')

# Convert marks to numpy
M_train = to_numpy(M_train)
M_val   = to_numpy(M_val)
M_test  = to_numpy(M_test)

train_ds = SequenceDatasetOverlap(X_train_final, M_train, y_train_label, seq_len, num_idxs, cat_idxs)
val_ds   = SequenceDatasetOverlap(X_val_final,   M_val,   y_val_label,   seq_len, num_idxs, cat_idxs)
test_ds  = SequenceDatasetOverlap(X_test_final,  M_test,  y_test_label,  seq_len, num_idxs, cat_idxs)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

del train_ds, val_ds, test_ds
gc.collect()

---Model Training, Validation and Testing Starts ---
Split Train, Val, Test
Rescaling 1 numeric features
Processing 1 categorical features
Data Loading


594

In [17]:
from training import run_classifier_training_pipeline
from utility import EarlyStopping
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 1. Define metadata correctly
# Labels: max(y) + 1 gives the number of classes (e.g., 0 and 1 -> 2 classes)
unique_classes, counts = np.unique(df[col_resp].values, return_counts=True)
num_classes = len(unique_classes)
# num_classes = int(np.max(train_loader.dataset.y) + 1)
n_num = int(len(train_loader.dataset.num_indices))
n_cat = int(len(train_loader.dataset.cat_indices))

# Dynamic cardinality for categorical embeddings
if n_cat > 0:
    cat_data = train_loader.dataset.X[:, train_loader.dataset.cat_indices]
    cat_card_list = [int(np.max(cat_data[:, j]) + 1) for j in range(n_cat)]
else:
    cat_card_list = []

total_samples = len(train_loader.dataset)
num_batches = len(train_loader)

## Getting dataset name
dataset_name = '../../data/TON_IoT/IoT_Fridge.csv'.split('IoT_')[1].rsplit('.csv', 1)[0]

print(f"--- Starting Training for Dataset {dataset_name} ---")
# Changed 'num_features' to 'n_num + n_cat' to avoid NameError
print(f"Features: {n_num + n_cat}, Samples: {total_samples}, Batches: {num_batches}, Classes: {num_classes}")   

# Setting baseline
baseline_stats = get_system_stats(device)

# 2. Initialize Model (Ensure the class cell has been RUN before this)
model = InformerClassifier(
    enc_in_num=n_num, 
    enc_in_cat=cat_card_list,
    num_classes=num_classes,
    d_model=512,
    n_heads=8,
    e_layers=3,
    d_ff=512,
    dropout=0.1,
    distil=True,
    attn='prob',
    freq='t'
).to(device)

# Calculate Class weights
# counts = df['type'].value_counts().sort_index().values
# class_sample_counts = np.array([len(np.where(train_loader.dataset.y == t)[0]) for t in np.unique(y_train)])
weights = len(train_loader.dataset.y) / (num_classes * counts)
class_weights = torch.FloatTensor(weights).to(device)

# CrossEntropy is standard for Supervised Classification
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

## Storing model parameters
model_configs = {
    'n_num': n_num,
    'cat_card_list': cat_card_list,
    'num_classes': num_classes,
    'dataset_name': dataset_name
}

# 3. Call the pipeline
# Note: Ensure you are calling 'run_classifier_training_pipeline' (supervised)
stats = run_classifier_training_pipeline(
    model=model,
    train_loader=train_loader,
    # val_loader=val_loader,
    num_epochs=num_epochs,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    baseline_stats=baseline_stats
)

print(f'Training of dataset Completed')

# Saving outputs
trained_model = model.to('cpu') 
final_stats = stats[-1].copy()
final_stats['dataset_name'] = dataset_name

Using device: cuda
--- Starting Training for Dataset Fridge ---
Features: 2, Samples: 55, Batches: 2, Classes: 2
Epoch 1/50 | Loss: 0.4100 | Acc: 0.9273 | Time: 0.44s
Epoch 2/50 | Loss: 0.0956 | Acc: 1.0000 | Time: 0.02s
Epoch 3/50 | Loss: 0.0250 | Acc: 1.0000 | Time: 0.02s
Epoch 4/50 | Loss: 0.0090 | Acc: 1.0000 | Time: 0.02s
Epoch 5/50 | Loss: 0.0042 | Acc: 1.0000 | Time: 0.02s
Epoch 6/50 | Loss: 0.0022 | Acc: 1.0000 | Time: 0.02s
Epoch 7/50 | Loss: 0.0014 | Acc: 1.0000 | Time: 0.02s
Epoch 8/50 | Loss: 0.0011 | Acc: 1.0000 | Time: 0.02s
Epoch 9/50 | Loss: 0.0007 | Acc: 1.0000 | Time: 0.02s
Epoch 10/50 | Loss: 0.0005 | Acc: 1.0000 | Time: 0.02s
Epoch 11/50 | Loss: 0.0004 | Acc: 1.0000 | Time: 0.02s
Epoch 12/50 | Loss: 0.0004 | Acc: 1.0000 | Time: 0.02s
Epoch 13/50 | Loss: 0.0003 | Acc: 1.0000 | Time: 0.02s
Epoch 14/50 | Loss: 0.0003 | Acc: 1.0000 | Time: 0.02s
Epoch 15/50 | Loss: 0.0002 | Acc: 1.0000 | Time: 0.02s
Epoch 16/50 | Loss: 0.0002 | Acc: 1.0000 | Time: 0.02s
Epoch 17/50 | Lo

In [18]:
# Saving performance
# Create a DataFrame where each epoch is a row
df_stats = pd.DataFrame(stats)

# Add the dataset name as a column so you can identify it later
df_stats['dataset'] = dataset_name

# Display the last 5 epochs to check progress
display(df_stats.tail())
print(f"Overall Training accuracy is: {np.round(df_stats['accuracy'].mean(),2)}")

# Save to CSV
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_path = f"../output/training_stats_{dataset_name}_{timestamp}.csv"
df_stats.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")

,epoch,avg_loss,accuracy,dataset
45,46,0.000018,1.0,Fridge
46,47,0.000017,1.0,Fridge
47,48,0.000013,1.0,Fridge
48,49,0.000013,1.0,Fridge
49,50,0.000012,1.0,Fridge


Overall Training accuracy is: 1.0
Results saved to: ../output/training_stats_Fridge_2026-04-12_08-45-16.csv


# Model Validation and Testing

In [26]:
test_stats

{'accuracy': 0.3785310734463277,
 'balanced_accuracy': np.float64(0.5),
 'f1_macro': 0.27459016393442626,
 'f1_weighted': 0.20788181902380296,
 'precision_attack': 0.0,
 'recall_attack': 0.0,
 'f1_attack': 0.0,
 'confusion_matrix': array([[ 67,   0],
        [110,   0]]),
 'report': {'0': {'precision': 0.3785310734463277,
   'recall': 1.0,
   'f1-score': 0.5491803278688525,
   'support': 67.0},
  '1': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 110.0},
  'accuracy': 0.3785310734463277,
  'macro avg': {'precision': 0.18926553672316385,
   'recall': 0.5,
   'f1-score': 0.27459016393442626,
   'support': 177.0},
  'weighted avg': {'precision': 0.14328577356442912,
   'recall': 0.3785310734463277,
   'f1-score': 0.20788181902380296,
   'support': 177.0}}}

In [20]:
from validation import run_classifier_evaluation_pipeline
from testing import run_classifier_testing_pipeline
from utility import select_best_threshold, model_perf, model_perf_plot
import torch
import pandas as pd

# --- CRITICAL STEP: Load the best model found by Early Stopping ---
# This ensures we aren't testing a model that started overfitting in the final epochs
# try:
#     trained_model.load_state_dict(torch.load('../models/best_model.pt'))
#     print("✅ Successfully loaded the best model weights for testing.")
# except FileNotFoundError:
#     print("⚠️ 'best_model.pt' not found. Using the current model weights instead.")

# 1. Validation 
# Still useful to verify the saved model's performance on the val set
val_loss, val_acc, val_f1, (val_probs, val_labels) = run_classifier_evaluation_pipeline(
    model.to(device), 
    val_loader, 
    device
)

best_thresh = select_best_threshold(val_probs, val_labels)

# 2. Testing (Final Unseen Data)
test_stats, y_pred, y_true, test_probs, anomaly_scores= run_classifier_testing_pipeline(
    trained_model.to(device), test_loader, device, threshold=best_thresh
)

# 3. Metric Extraction
report = test_stats['report']

# ToN_IoT targets: '1' is usually the attack class. 
target_key = '1' if '1' in report else 1 

result_entry = {
    "dataset": dataset_name,
    "val_acc": val_acc,
    "test_acc": test_stats["accuracy"],
    "f1_weighted": test_stats["f1_score"],
    "precision_attack": report[target_key]['precision'] if target_key in report else 0,
    "recall_attack": report[target_key]['recall'] if target_key in report else 0
}


# 4. Display & Save
results_df = pd.DataFrame([result_entry])
display(results_df)

# Add to your historical log
results_df.to_csv(f"../output/final_results_{dataset_name}.csv", index=False)

Validation Complete | Loss: 11.1276 | Acc: 0.0000 | F1: 0.0000
Best Threshold Found: 1.0000
Max F1-Score at this threshold: 0.0000
------------------------------
OVERALL ACCURACY:  0.3785
BALANCED ACCURACY: 0.5000 (True performance)
MACRO F1 SCORE:    0.2746
------------------------------
ATTACK RECALL:     0.0000 (Detection Rate)
ATTACK PRECISION:  0.0000
------------------------------


/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


KeyError: 'f1_score'

In [19]:
y_pred
y_true

array([1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1])

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [33]:
errors = val_probs[:, 1]
new_label = (np.abs((errors-errors.mean())/errors.std())>(2*errors.std())).astype(int)
# new_label = (np.abs(0.6745 * ((x := val_probs[:, 1]) - (m := np.median(x))) / (np.median(np.abs(x - m)) + 1e-8)) > 2.5).astype(int)
f1_new = f1_score(y_true, new_label, average='weighted')
acc_new = accuracy_score(y_true, new_label)
prec_new = precision_score(y_true, new_label, average='weighted')
rec_new = recall_score(y_true, new_label, average='weighted')

report_new = classification_report(y_true, new_label, output_dict=True)
conf_matrix_new = confusion_matrix(y_true, new_label)

/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu

In [34]:
model_perf(orig=y_true, pred=new_label)

Confusion Matrix:
[[ 0  0]
 [23 65]]
Type 1 Error (FPR): nan
Type 2 Error (FNR): 0.2614
Accuracy: 0.7386

Classification Report:
              precision    recall  f1-score   support

      Normal       0.00      0.00      0.00         0
    Abnormal       1.00      0.74      0.85        88

    accuracy                           0.74        88
   macro avg       0.50      0.37      0.42        88
weighted avg       1.00      0.74      0.85        88



/home/evgenia/Antoni/Roland/2026/IoT_Attacks_Detection/code/utility.py:372: RuntimeWarning: invalid value encountered in scalar divide
  type_1_error = FP / (FP + TN)  # Probability of a False Alarm
/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/evgenia/anaconda3/envs/torchNew/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: 

{'precision': np.float64(1.0),
 'accuracy': np.float64(0.7386363636363636),
 'recall': np.float64(0.7386363636363636),
 'f1_score': np.float64(0.8496732026143791),
 'auroc': None,
 'type_1_error': np.float64(nan)}

In [22]:
### Saving Model
output_dir='../output/'
os.makedirs(output_dir, exist_ok=True)

# Timestamp
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

## Extract dataset name
dataset_name = model_configs['dataset_name']

# Inside your loop:
save_path = f"../models/informer_{dataset_name}_{timestamp}.pth"
torch.save(model_configs, save_path)
print(f"Checkpoint saved to {save_path}")

io_data = {
    'true_labels':y_true,
    'pred_labels':y_pred,
    # 'thresholds':threshold,
    'stats':result_entry
}

perf_filename = f"informer_perf_stats_iodata_{dataset_name}_{timestamp}.pkl"
full_perf_path = os.path.join(output_dir, perf_filename)
joblib.dump(io_data, full_perf_path)

Checkpoint saved to ../models/informer_Fridge_2026-03-23_08-08-26.pth


['../output/informer_perf_stats_iodata_Fridge_2026-03-23_08-08-26.pkl']

## Quasi-dynamic threshold

In [23]:
# model_perf = pd.read_pickle("../../output/w_labels/informer_perf_stats_iodata_Fridge_2026-03-16_08-20-26.pkl")
model_path="/home/mfondoum/Documents/PhD/informer_Thermostat_2026-03-12_06-59-24.pth"

In [24]:
model_path

'/home/mfondoum/Documents/PhD/informer_Thermostat_2026-03-12_06-59-24.pth'